Notebook creates & saves basic hypothesis sets for PRC, plus performs wald test. 

In [1]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster
import pandas as pd

2025-11-14 09:53:24.645636: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-14 09:53:25.552022: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/code-server/4.91.1/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

In [2]:
from pathlib import Path

In [3]:
DATA_ROOT=Path("/gpfs/gibbs/pi/reilly/tabula_data")
path=DATA_ROOT/"simulated"
name="shendure_calibrated_sim_with_orthos_20251008"

In [4]:
demo_counts=scm.scMPRA_data.from_parquet(path/name/"scMPRA/0.scmpra")
demo_counts.ortho_filter()

scMPRAforge: INFO: Dropped 618 of 2080 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [5]:
# all CREs within each cell type, vs the 'reference' negative control
hs_all_ct = scm.make_all_by_celltype_hypotheses(
    counts=demo_counts,
    reference_cre="reference",
    meta="emvar_screen",
)

# all cell types for each CRE, vs the dataset’s baseline cell type
hs_all_cre = scm.make_all_by_cre_hypotheses(
    counts=demo_counts,
    reference_cell_type="reference",  # will be normalized to 'reference'
    meta="cell_specificity",
)

In [6]:
local=False
if local:
    cluster=LocalCluster(memory_limit='48G')
    client = Client(cluster)
else:
    cluster=SLURMCluster(
        cores=8,#cores per slurm job
        memory="80G",#memory per slurm job
        processes=1,#dask workers per slurm job
        job_extra_directives=["-p ycga", 
            f"--job-name=simclust_worker",
            f"--time=1:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=3)
    client = Client(cluster,
            timeout=f"{10*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s",  # Worker heartbeat interval,
        )

In [7]:
from dask.distributed import Semaphore, as_completed, get_client

In [8]:
ortho_root=path/name/"orthos_with_precomputed_wald_erin_numerical_stability_0"
output_root=path/name/"results"
output_root.mkdir(exist_ok=True,parents=True)
input_ortho_names=[path.name for path in ortho_root.iterdir()]

Semaphore(max_leases=3, name="test")

def compute_one_wald(input_root, name, output_root, hypothesis_set, hypothesis_set_name, test_type):
    sem = Semaphore(name="test")
    with sem:
        client=get_client()
        ortho_oi=scm.ortho.load(client=client,
                                    path=input_root,
                                    name=name)
        runner = scm.HypothesisTester(test_type)
        output_short=Path(output_root)/hypothesis_set_name/test_type
        output_short.mkdir(exist_ok=True,parents=True)
        runner.run(hypothesis_set, ortho_oi, client).to_tsv(output_short/name)



In [27]:
input_ortho_names

['0', '1', '2', '3', '4']

In [43]:
client=get_client()
ortho_oi=scm.ortho.load(client=client,
                                    path=ortho_root,
                                    name=input_ortho_names[0])

ortho_oi.by_cell_type.model['Mesoderm'].result()

{'llf_total': nan,
 'llfs': array([nan]),
 'aic_total': nan,
 'aics': array([nan]),
 'df_model_total': 184,
 'df': 184,
 'weights': {'x_mu': Intercept                                                           NaN
  C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8172]     NaN
  C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8174]     NaN
  C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8175]     NaN
  C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8179]     NaN
                                                                       ..
  C(cre_id, contr.treatment(base='reference'))[T.Txndc12_chr4_7973]   NaN
  C(cre_id, contr.treatment(base='reference'))[T.Txndc12_chr4_7978]   NaN
  C(cre_id, contr.treatment(base='reference'))[T.eef1aP]              NaN
  C(cre_id, contr.treatment(base='reference'))[T.pgk1P]               NaN
  C(cre_id, contr.treatment(base='reference'))[T.ubcP]                NaN
  Length: 177, dtype: float32,
  'x_pi': C(rep_

In [42]:
for rep in (input_ortho_names):
    print("Simulation replicate: %s" % rep)
    ortho_oi=scm.ortho.load(client=client,
                                    path=ortho_root,
                                    name=rep)
    for ct in ortho_oi.by_cell_type.model.keys():
        print("%s llf_total: %s" % (ct, ortho_oi.by_cell_type.model[ct].result()['llf_total']))

    print()

Simulation replicate: 0
Cardiomyocytes llf_total: -29359.958255389123
EpiblastPrimitiveStreak llf_total: -222074.86762518808
ExEndodermParietal llf_total: -322061.68298781617
ExEndodermVisceral llf_total: -156219.62599534635
Haematoendothelial llf_total: -47923.298563371645
Mesoderm llf_total: nan
NeuroectodermBrain llf_total: -303378.1891246969
NeuroectodermRostral llf_total: -60295.0163888259
SurfaceEctoderm llf_total: -168038.30045224493
reference llf_total: -605025.7000021078

Simulation replicate: 1
Cardiomyocytes llf_total: -28647.040158434538
EpiblastPrimitiveStreak llf_total: nan
ExEndodermParietal llf_total: -321301.3699642923
ExEndodermVisceral llf_total: -157662.26673798636
Haematoendothelial llf_total: -48387.60003270302
Mesoderm llf_total: -290035.30614223797
NeuroectodermBrain llf_total: -299767.1379107144
NeuroectodermRostral llf_total: -59954.260172023845
SurfaceEctoderm llf_total: nan
reference llf_total: -613713.4881491438

Simulation replicate: 2
Cardiomyocytes llf_t

nan

In [9]:
#hs_all_ct
futures = [client.submit(compute_one_wald,
                        input_root=ortho_root,
                        name=name_oi,
                        output_root=output_root,
                        hypothesis_set=hs_all_ct,
                        hypothesis_set_name="hs_all_ct",
                        test_type="wald") for name_oi in input_ortho_names]

In [10]:
#hs_all_cre
futures = [client.submit(compute_one_wald,
                        input_root=ortho_root,
                        name=name_oi,
                        output_root=output_root,
                        hypothesis_set=hs_all_cre,
                        hypothesis_set_name="hs_all_cre",
                        test_type="wald") for name_oi in input_ortho_names]

In [11]:
scmpradat_root=path/name/"scMPRA"
def compute_one_mwu(input_root, name, output_root, hypothesis_set, hypothesis_set_name, test_type):
    sem = Semaphore(name="test")
    with sem:
        client=get_client()
        dat=scm.scMPRA_data.from_parquet(scmpradat_root/Path(name).with_suffix(".scmpra"))
        dat.ortho_filter()
        runner = scm.HypothesisTester(test_type)
        output_short=Path(output_root)/hypothesis_set_name/test_type
        output_short.mkdir(exist_ok=True,parents=True)
        runner.run(hypothesis_set, dat, client).to_tsv(output_short/name)

In [12]:
futures_mwu = [client.submit(compute_one_mwu,
                        input_root=ortho_root,
                        name=name_oi,
                        output_root=output_root,
                        hypothesis_set=hs_all_ct,
                        hypothesis_set_name="hs_all_ct",
                        test_type="mwu") for name_oi in input_ortho_names]

In [13]:
futures_mwu

[<Future: finished, type: NoneType, key: compute_one_mwu-d5d75dcb76189276ec91b688821d49ce>,
 <Future: finished, type: NoneType, key: compute_one_mwu-096d851457dff35e0b8dc213a235935b>,
 <Future: finished, type: NoneType, key: compute_one_mwu-8aeefbf6a642a51b6f9ee2abd4712204>,
 <Future: finished, type: NoneType, key: compute_one_mwu-a7f776c88d21f353f3da27defcd5b4a2>,
 <Future: finished, type: NoneType, key: compute_one_mwu-0841e1596d74f2e52d3bf1ec8b77547d>]

In [14]:
futures_mwu = [client.submit(compute_one_mwu,
                        input_root=ortho_root,
                        name=name_oi,
                        output_root=output_root,
                        hypothesis_set=hs_all_cre,
                        hypothesis_set_name="hs_all_cre",
                        test_type="mwu") for name_oi in input_ortho_names]

In [15]:
client.close()
cluster.close()

In [21]:
test_particle=scm.ortho.load(client=client,
                                    path=ortho_root,
                                    name='0')

In [34]:
ortho_oi.wald_precomp.by_cell_type['Cardiomyocytes'].result().debug_msg

'ok'

In [37]:
def get_test_particle(i):
    return scm.ortho.load(client=client,
                                    path=ortho_root,
                                    name=i)
def track_errs(test_particle, repi):
    names=[]
    errors=[]
    for name in test_particle.wald_precomp.by_cell_type:
        names.append(name)
        errors.append(test_particle.wald_precomp.by_cell_type[name].result().debug_msg)
    pd.DataFrame({"name":names,"debug":errors}).to_csv("by_cell_types_rep%s.tsv" % repi,sep="\t")

    names=[]
    errors=[]
    for name in test_particle.wald_precomp.by_cre:
        names.append(name)
        errors.append(test_particle.wald_precomp.by_cre[name].result().debug_msg)
    pd.DataFrame({"name":names,"debug":errors}).to_csv("by_cre_rep%s.tsv" % repi,sep="\t")

In [40]:
rep0, rep1, rep2, rep3, rep4 = [get_test_particle(str(repi)) for repi in [0,1,2,3,4]]

In [43]:
track_errs(rep0, 0)
track_errs(rep1, 1)
track_errs(rep2, 2)
track_errs(rep3, 3)
track_errs(rep4, 4)

In [48]:
dir(rep0.wald_precomp.by_cell_type['Cardiomyocytes'].result())

['__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slotnames__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 'cov_nb',
 'debug_msg',
 'k_nb',
 'name_to_idx',
 'se_x_mu',
 'xmu_names']